# 8.9 — NumPy Blocks for Full-Batch Gradient Descent

**Chapter 8, section 8.1.3** (the paragraph on `mapPartitions`) **and section 8.2.3** (the gradient
descent implementation).

**The question this notebook answers:** every gradient-descent loop in this chapter stores the
training set as an RDD of `(y, x)` pairs, one pair per training example. What does that storage
cost on every pass, and what changes if each partition is stored as **one** NumPy vector `y` and
**one** NumPy matrix `X` instead?

**The short answer:** with one pair per row, every pass runs Python code once per row — a million
rows means a million tiny `np.dot` calls and a million small arrays, on every iteration. With one
block per partition, a pass is a few matrix products per partition, which NumPy runs in compiled
loops. The arithmetic is the same, the parameters come out the same, and the pass is many times
faster. The conversion is paid once, before the loop, and the loop repays it within a few
iterations.

In [1]:
# --- CS-777 session setup ------------------------------------------------
import os, tempfile, time
import numpy as np
import pandas as pd
from pyspark.sql import SparkSession, functions as F

SCRATCH = os.environ.get("CS777_SCRATCH", os.path.join(tempfile.gettempdir(), "cs777"))
os.makedirs(SCRATCH, exist_ok=True)

spark = (SparkSession.builder
         .appName("CS777-8.9")
         .master("local[4]")
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.sql.warehouse.dir", os.path.join(SCRATCH, "warehouse"))
         .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("ERROR")

print("Spark", spark.version)

Spark 4.2.0


---

## 1. Where the time goes in a row-by-row pass

In the chapter's loops the training set is an RDD whose elements are pairs `(y, x)`, with `x` a
NumPy array of length $d$ that starts with the constant $1$ for the intercept. One pass computes
the loss and the gradient at the current parameters $\theta$,

$$
L(\theta) = \tfrac12 \sum_i \bigl(y_i - x_i^{\mathsf T}\theta\bigr)^2,
\qquad
\nabla L(\theta) = -\sum_i x_i \bigl(y_i - x_i^{\mathsf T}\theta\bigr),
$$

and `treeAggregate` folds one example at a time into the accumulator. So for **every row, on every
pass**, the Python worker unpickles one tuple and one small array, calls `np.dot` on two vectors of
length four, and allocates two or three more four-element arrays. Each of those calls is cheap;
a million of them per pass are not. The time goes into the interpreter and the object overhead,
not into the arithmetic, which is only a few multiplications per row.

A **block** removes that overhead by changing the unit of work. Each partition is converted
**once** into a vector `y` of its labels and a matrix `X` of its feature vectors, one row per
example. The same loss and gradient are then

$$
r = y - X\theta, \qquad L = \tfrac12\, r^{\mathsf T} r, \qquad \nabla L = -X^{\mathsf T} r,
$$

three NumPy calls per partition, each running a compiled loop over all the rows at once.

The conversion is not free: it reads every row once and builds the arrays. That is why the block
RDD is **cached**. Gradient descent reads the training set once per iteration, so a conversion
done once before the loop is repaid on every iteration after it.

---

## 2. One dataset, two RDDs built from it

The data follow the chapter's own synthetic model from section 8.2.3: an intercept of $5$ and
coefficients $(-1, 1, 2)$ on three uniform features, plus a little noise. The effect this notebook
measures is per-row overhead, which is invisible on a few thousand rows, so it uses **a million
rows** in four partitions. They are generated by Spark itself with `range` and `rand`, so nothing
is shipped from the driver, and cached as a DataFrame. Both RDDs below are built from this same
cached DataFrame, so both start from the same place.

In [2]:
N, PARTS = 1_000_000, 4
COLS = ["one", "x1", "x2", "x3"]                 # the model's columns, in parameter order

df = (spark.range(N, numPartitions=PARTS)
      .select(F.lit(1.0).alias("one"),
              F.rand(seed=1).alias("x1"), F.rand(seed=2).alias("x2"), F.rand(seed=3).alias("x3"),
              F.randn(seed=4).alias("noise"))
      .select((5 - F.col("x1") + F.col("x2") + 2 * F.col("x3") + 0.1 * F.col("noise")).alias("y"),
              *COLS)
      .cache())
print("rows:", df.count(), "  partitions:", df.rdd.getNumPartitions())
df.show(3)

rows: 1000000   partitions: 4
+-----------------+---+------------------+-------------------+-------------------+
|                y|one|                x1|                 x2|                 x3|
+-----------------+---+------------------+-------------------+-------------------+
|5.298686607301138|1.0|0.6363787615254752| 0.5311207224659675|0.25738143505962285|
|6.003584262215113|1.0|0.5993846534021868| 0.2861372051669987| 0.6698885713796182|
|7.222393097157727|1.0| 0.134842710012538|0.49443063728956627| 0.9416651198480125|
+-----------------+---+------------------+-------------------+-------------------+
only showing top 3 rows


**The row RDD** is the representation the rest of the chapter uses: one `(y, x)` pair per row.

In [3]:
t0 = time.perf_counter()
rows = df.rdd.map(lambda r: (r[0], np.array(r[1:]))).cache()
rows.count()                                     # materialize the cache
build_rows = time.perf_counter() - t0

print(rows.first())
print(f"built and cached in {build_rows:.2f} s")

(5.298686607301138, array([1.        , 0.63637876, 0.53112072, 0.25738144]))
built and cached in 1.32 s


**The block RDD** uses `mapPartitions`, which hands the function an iterator over a whole
partition. A `Row` is a tuple, so `np.array(list(it))` turns the partition into one matrix with
one row per example; column 0 is the label and the rest are the features. The function yields a
single `(y, X)` pair, so the resulting RDD has **exactly one element per partition**.

In [4]:
def to_block(it):
    a = np.array(list(it), dtype=float)          # the whole partition, n_p x (1 + d)
    if len(a):                                   # an empty partition yields no block
        yield a[:, 0].copy(), np.ascontiguousarray(a[:, 1:])

t0 = time.perf_counter()
blocks = df.rdd.mapPartitions(to_block).cache()
blocks.count()                                   # materialize the cache
build_blocks = time.perf_counter() - t0

shapes = blocks.map(lambda b: (b[1].shape, (b[0].nbytes + b[1].nbytes) / 1e6)).collect()
print("elements in the block RDD:", blocks.count())
for (shape, mb) in shapes:
    print(f"  block X {shape},  {mb:.1f} MB of arrays")
print(f"built and cached in {build_blocks:.2f} s")

elements in the block RDD: 4
  block X (250000, 4),  10.0 MB of arrays
  block X (250000, 4),  10.0 MB of arrays
  block X (250000, 4),  10.0 MB of arrays
  block X (250000, 4),  10.0 MB of arrays
built and cached in 1.91 s


---

## 3. One pass, three ways

All three return the chapter's accumulator: the summed loss, the summed gradient, and the count.
The zero value is `(0.0, np.zeros(d), 0)` and the merge is componentwise addition.

**Rows.** `seqOp` folds in one example — Python runs this function a million times per pass.

In [5]:
D = len(COLS)
ZERO = (0.0, np.zeros(D), 0)
add = lambda a, b: (a[0] + b[0], a[1] + b[1], a[2] + b[2])


def pass_rows(theta):
    def seq(acc, pair):
        y, x = pair
        r = y - np.dot(x, theta)
        return acc[0] + 0.5 * r * r, acc[1] - x * r, acc[2] + 1
    return rows.treeAggregate(ZERO, seq, add)

**Blocks.** `map` turns each block into its own triple with three matrix operations. Each partition
now holds one element, so each partition's `seqOp` runs once, and it is the same `add` used to
merge partitions.

In [6]:
def pass_blocks(theta):
    def one(yX):
        y, X = yX
        r = y - X @ theta
        return 0.5 * float(r @ r), -(X.T @ r), len(y)
    return blocks.map(one).treeAggregate(ZERO, add, add)

**DataFrame.** For reference, the relational version from notebook 8.8: one `agg`, no Python per
row, with the current $\theta$ written into the query as literals.

In [7]:
def pass_df(theta):
    pred = sum(F.lit(float(t)) * F.col(c) for t, c in zip(theta, COLS))
    r = F.col("y") - pred
    row = df.agg(F.sum(0.5 * r * r).alias("loss"),
                 *[F.sum(-r * F.col(c)).alias(c) for c in COLS],
                 F.count(F.lit(1)).alias("n")).first()
    return row["loss"], np.array([row[c] for c in COLS]), row["n"]

The three must agree before any of them is timed. They are compared at an arbitrary $\theta$, so
that neither the loss nor the gradient is trivially zero.

In [8]:
theta_test = np.array([1.0, 2.0, 3.0, 4.0])
results = {"rows": pass_rows(theta_test), "blocks": pass_blocks(theta_test),
           "DataFrame": pass_df(theta_test)}

for name, (loss, grad, n) in results.items():
    print(f"{name:10} loss={loss:.6f}  grad={np.round(grad, 4)}  n={n}")

ref = results["rows"]
for name, (loss, grad, n) in results.items():
    assert n == N and np.isclose(loss, ref[0]) and np.allclose(grad, ref[1]), name
print("\nall three passes agree: same loss, same gradient, same count")

rows       loss=835440.006876  grad=[-4.98444327e+05  1.72188700e+02 -8.32367683e+04 -8.35572188e+04]  n=1000000
blocks     loss=835440.006876  grad=[-4.98444327e+05  1.72188700e+02 -8.32367683e+04 -8.35572188e+04]  n=1000000
DataFrame  loss=835440.006876  grad=[-4.98444327e+05  1.72188700e+02 -8.32367683e+04 -8.35572188e+04]  n=1000000

all three passes agree: same loss, same gradient, same count


Now the time for one pass. Everything is cached and materialized already; each figure is the
median of three passes after one untimed warm-up. Timings are machine-dependent (this notebook
runs on `local[4]`); the ratios are the point.

In [9]:
def time_it(fn, repeats=3):
    fn()                                         # warm-up, not timed
    secs = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        fn()
        secs.append(time.perf_counter() - t0)
    return float(np.median(secs))


per_pass = {name: time_it(lambda f=f: f(theta_test))
            for name, f in (("rows", pass_rows), ("blocks", pass_blocks), ("DataFrame", pass_df))}

table = pd.DataFrame({"seconds per pass": per_pass}).round(4)
table["x faster than rows"] = (per_pass["rows"] / table["seconds per pass"]).round(1)
print(table.to_string())

           seconds per pass  x faster than rows
rows                 0.3553                 1.0
blocks               0.0277                12.8
DataFrame            0.0617                 5.8


The row version is the slowest by a wide margin, and the reason is the one section 1 gave: it
executes Python once per row. The block version does the same arithmetic in a handful of calls per
partition. The DataFrame version also avoids per-row Python, by compiling the aggregate to JVM
code. It is also far ahead of the rows. On this machine it trails the blocks; section 5 shows
that on a cluster the order of those two was reversed.

---

## 4. Full-batch gradient descent on blocks

The training loop does not care how a pass is computed: it only needs a function that returns
`(loss, gradient, count)` for the current $\theta$. So one loop drives all three versions. It
starts from $\theta = 0$ and uses $\eta = 1$ for ten iterations, the setting of section 8.2.3,
where the chapter shows the coefficients oscillating while the cost falls.

In [10]:
def gradient_descent(pass_fn, eta=1.0, iterations=10, trace=False):
    theta = np.zeros(D)
    for i in range(iterations):
        loss, grad, n = pass_fn(theta)
        theta = theta - eta * grad / n
        if trace:
            print(f"{i} Cost={loss / n:.5f} theta={theta.round(5)}")
    return theta


theta_blocks = gradient_descent(pass_blocks, trace=True)

0 Cost=18.25620 theta=[6.00025 2.91864 3.08388 3.16758]
1 Cost=11.40133 theta=[1.4132  0.29726 0.61662 0.77693]
2 Cost=7.14235 theta=[5.15452 2.06099 2.51954 2.75013]
3 Cost=4.49361 theta=[2.33337 0.39457 0.98234 1.27727]
4 Cost=2.84405 theta=[4.67267 1.44876 2.1537  2.50764]
5 Cost=1.81479 theta=[2.94396 0.3799  1.19326 1.60123]
6 Cost=1.17086 theta=[4.41248 0.99967 1.91161 2.36914]
7 Cost=0.76653 theta=[3.35902 0.30605 1.30899 1.8119 ]


8 Cost=0.51136 theta=[4.28618 0.66114 1.74699 2.29151]


9 Cost=0.34922 theta=[3.64957 0.20428 1.36652 1.94915]


The same loop on the other two representations, each timed as a whole run. The parameters must
match: the three versions differ only in the order in which floating-point sums are formed, so
they agree to rounding error and no further.

In [11]:
fits, secs = {}, {}
for name, f in (("rows", pass_rows), ("blocks", pass_blocks), ("DataFrame", pass_df)):
    t0 = time.perf_counter()
    fits[name] = gradient_descent(f)
    secs[name] = time.perf_counter() - t0

for name in ("rows", "DataFrame"):
    diff = np.max(np.abs(fits[name] - fits["blocks"]))
    print(f"max |theta_{name} - theta_blocks| = {diff:.2e}")
    assert diff < 1e-8
print("theta after 10 iterations:", fits["blocks"].round(5))

max |theta_rows - theta_blocks| = 3.40e-14
max |theta_DataFrame - theta_blocks| = 3.40e-14
theta after 10 iterations: [3.64957 0.20428 1.36652 1.94915]


In [12]:
build = {"rows": build_rows, "blocks": build_blocks, "DataFrame": float("nan")}
summary = pd.DataFrame({
    "build once (s)": build,
    "per pass (s)": per_pass,
    "10 iterations (s)": secs,
}).round(3)
summary["build + 10 iterations (s)"] = (summary["build once (s)"] + summary["10 iterations (s)"]).round(2)
print(summary.to_string())

saved_per_pass = per_pass["rows"] - per_pass["blocks"]
extra_build = build_blocks - build_rows
print(f"\nextra cost of building blocks instead of rows: {extra_build:+.2f} s")
print(f"time saved per pass by blocks:                 {saved_per_pass:.3f} s")
print(f"blocks come out ahead after about {max(extra_build, 0) / saved_per_pass:.1f} passes")
print("\nThe DataFrame is built once as the source of both RDDs; its build is not charged to it here.")

           build once (s)  per pass (s)  10 iterations (s)  build + 10 iterations (s)
rows                1.324         0.355              3.574                       4.90
blocks              1.909         0.028              0.248                       2.16
DataFrame             NaN         0.062              0.854                        NaN

extra cost of building blocks instead of rows: +0.59 s
time saved per pass by blocks:                 0.328 s
blocks come out ahead after about 1.8 passes

The DataFrame is built once as the source of both RDDs; its build is not charged to it here.


Two things to read in the table. The **build** of the block RDD costs somewhat more than the
build of the row RDD, but it is the same order of magnitude — both read every row of the DataFrame
once — so the conversion is not where the difference lies. The **passes** are where it lies: the
block loop's ten iterations take a small fraction of the row loop's, and the break-even line shows
the extra conversion work repaid within a few passes. A real training run makes tens or hundreds of passes, and every one
of them is charged at the per-pass rate.

---

## 5. What a block costs in memory

A block trades per-row overhead for size, and three consequences follow.

**A block holds a whole partition in one Python process.** The arrays themselves are compact —
eight bytes per number, as the build cell printed. But `to_block` first materializes
`list(it)`, a Python list of a quarter of a million `Row` objects, and only then copies it into an
array. At its peak the worker holds both, and the list of objects is several times larger than the
array it becomes. On a large dataset with few partitions that temporary can exhaust the Python
worker's memory. The remedy is **more, smaller partitions**, for example `df.repartition(k)` before
`mapPartitions`, so that each block stays a manageable size. Smaller blocks mean more tasks, so
the pass costs somewhat more, but a block of sixty thousand rows is still a single matrix product
and the pass stays far cheaper than the row version's:

In [13]:
blocks16 = df.repartition(16).rdd.mapPartitions(to_block).cache()
sizes = blocks16.map(lambda b: len(b[0])).collect()
print(f"{len(sizes)} blocks, {min(sizes):,} to {max(sizes):,} rows each")


def pass_blocks16(theta):
    def one(yX):
        y, X = yX
        r = y - X @ theta
        return 0.5 * float(r @ r), -(X.T @ r), len(y)
    return blocks16.map(one).treeAggregate(ZERO, add, add)


assert np.allclose(pass_blocks16(theta_test)[1], results["rows"][1])
t16 = time_it(lambda: pass_blocks16(theta_test))
print(f"per pass:  4 blocks {per_pass['blocks']:.4f} s   16 blocks {t16:.4f} s   "
      f"rows {per_pass['rows']:.4f} s  (rows take {per_pass['rows'] / t16:.1f}x as long)")
_ = blocks16.unpersist()

16 blocks, 62,500 to 62,500 rows each


per pass:  4 blocks 0.0277 s   16 blocks 0.0486 s   rows 0.3553 s  (rows take 7.3x as long)


**A cached RDD of Python objects is stored pickled, and every pass unpickles it.** PySpark keeps
the cached partitions as serialized bytes in the JVM. On each pass the bytes are sent to a Python
worker and turned back into objects before the first line of `one` runs. For the row RDD that is a
million small objects per pass; for the blocks it is four large arrays, which is a far cheaper
thing to deserialize. The cell below times a pass that delivers the cached data to Python and then
does nothing with it.

In [14]:
empty = {"rows": time_it(lambda: rows.map(lambda _: 0).count()),
         "blocks": time_it(lambda: blocks.map(lambda _: 0).count())}
for name in ("rows", "blocks"):
    print(f"{name:7} delivering the cache to Python: {empty[name]:.4f} s   "
          f"({empty[name] / per_pass[name]:.0%} of a full pass)")

rows    delivering the cache to Python: 0.1667 s   (47% of a full pass)
blocks  delivering the cache to Python: 0.0237 s   (86% of a full pass)


For the rows, simply getting a million objects into Python is already a large share of the pass,
before any arithmetic. For the blocks it is a small absolute cost, but it is a large *share* of
what is left, because the matrix products themselves are so cheap. That remaining cost is the
price of running the computation in Python at all, and it is the one thing the DataFrame version
never pays: its cached data is columnar and is read in place by compiled JVM code.

**On a cluster the DataFrame stayed ahead.** The same three versions of the linear-regression pass
were also measured on Google Cloud (Dataproc Serverless, Spark 3.5, 6 executors of 4 cores each) on
a much larger training set: about 134 million rows in 47 partitions, so each block held roughly
2.9 million rows. That run is not part of this notebook; its figures are quoted here.

| one full pass, linear regression | seconds |
| -------------------------------- | ------- |
| row RDD | 64.5 |
| blocks | 3.2 |
| DataFrame | 1.65 |

For logistic regression one iteration took 4.2 s on blocks and 3.1 s on the DataFrame. So on the
cluster the blocks were about twenty times faster than the rows, as here, but between 1.3 and 2
times **slower** than the DataFrame — the opposite of the order on this laptop. The likely
causes are the ones this section has measured: every pass unpickles the whole cached block in a
Python worker, and blocks of millions of rows make that step large.

Building the blocks took 80 to 113 seconds there, a one-time cost, but it was also where the run
nearly failed: one executor was killed for running out of memory while building
2.9-million-row blocks next to the cached DataFrames, and Spark had to rerun its tasks. That is the
peak-memory problem described above, at full size, and it is why smaller blocks — more partitions,
via `repartition` — are the safe default on large data.

The fair summary: **blocks close most of the gap between the row RDD and the DataFrame.** Which of
blocks and DataFrame wins depends on the machine and on the block size; on this laptop the blocks
won, and on the cluster the DataFrame stayed faster. Both are far ahead of the row-by-row RDD
everywhere they were measured.

---

## 6. Takeaway

* **An RDD of per-row `(y, x)` pairs makes every pass run Python once per row.** The cost is the
  interpreter and the object overhead, not the arithmetic, and it is paid on every iteration.
* **Converting each partition once into a NumPy block** — a vector `y` and a matrix `X` — turns a
  pass into $r = y - X\theta$, $\tfrac12 r^{\mathsf T}r$ and $-X^{\mathsf T}r$ per partition, combined
  with the same `treeAggregate` and the same zero value `(0.0, np.zeros(d), 0)` the chapter uses.
  The loss, the gradient and the fitted parameters are the same as the row version's and the
  DataFrame version's, to rounding error.
* **Build once, cache, reuse.** The conversion costs about one read of the data. Gradient descent
  reads the data once per iteration, so the conversion is repaid within a few iterations.
* **Keep blocks a manageable size.** A block is a whole partition in one Python process, and the
  conversion briefly holds it twice. On large data, use more partitions so each block stays small;
  on the cluster run, building 2.9-million-row blocks cost an executor to an out-of-memory kill.
* **Blocks close most of the gap to the DataFrame, not all of it.** They were fastest on this
  laptop; on the cluster the DataFrame stayed faster, because the block version still pays to move
  and unpickle its data in Python on every pass.

This is the same idea section 8.1.3 mentions under `mapPartitions` — work on a whole partition at
once — taken one step further: the partition is not just processed together on each pass, it is
**stored** together, in the form NumPy computes on fastest.